In [2]:
%load_ext autoreload
%autoreload 2

from src.routed_agent import run_routed_agent


In [2]:

# Each should route differently
test_queries = [
    ("Hi, how are you today?", "chat"),
    ("What is RAG?", "search_only"),
    ("Summarize the fine-tuning chapter in 3 bullets.", "search_analyze/summarize"),
    ("What are the key terms in evaluation?", "search_analyze/extract_terms"),
]


In [ ]:

for q, expected in test_queries:
    print(f"\n{'='*70}")
    print(f"Q: {q}")
    print(f"Expected route: {expected}")
    print('='*70)
    result = run_routed_agent(q)
    print(f"Actual route: {result['query_type']} / {result['analysis_type']}")
    print(f"Answer preview: {result['final_answer'][:250]}...")


Q: Hi, how are you today?
Expected route: chat
→ Classified as: chat ()
Actual route: chat / 
Answer preview: Hey! I'm doing well, thanks for asking! I'm here and ready to help with whatever you need. How are you doing today?...

Q: What is RAG?
Expected route: search_only
→ Classified as: search_only ()
→ Search returned 4767 chars
Actual route: search_only / 
Answer preview: # What is RAG?

Based on the provided context, **RAG stands for Retrieval-Augmented Generation**.

According to the book (page 277):

> "RAG is a technique that enhances a model's generation by retrieving the relevant information from external memory...

Q: Summarize the fine-tuning chapter in 3 bullets.
Expected route: search_analyze/summarize
→ Classified as: search_analyze (summarize)
→ Search returned 3633 chars
→ Analysis (summarize): 360 chars
Actual route: search_analyze / summarize
Answer preview: # Summary of Fine-Tuning Chapter

Based on the provided context, here are 3 key points:

• **Prioritize eval

: 

In [3]:
# Each should route differently
test_queries = [
    ("Do you know anything about RAG?", "search_only"),
    ("Yo, quick question — what's fine-tuning?", "search_only"),
    ("What can you do?", "chat"),
    ("Explain RAG to me and give me the key vocabulary.", "search_analyze/summarize/extract_terms"),
]

In [4]:
for q, expected in test_queries:
    print(f"\n{'='*70}")
    print(f"Q: {q}")
    print(f"Expected route: {expected}")
    print('='*70)
    result = run_routed_agent(q)
    print(f"Actual route: {result['query_type']} / {result['analysis_type']}")
    print(f"Answer preview: {result['final_answer'][:250]}...")


Q: Do you know anything about RAG?
Expected route: search_only
→ Classified as: chat ()
Actual route: chat / 
Answer preview: # RAG (Retrieval-Augmented Generation)

Yes! RAG is a technique that combines information retrieval with text generation. Here's the basic idea:

## How it works:
1. **Retrieval** - When you ask a question, the system searches through a knowledge bas...

Q: Yo, quick question — what's fine-tuning?
Expected route: search_only
→ Classified as: search_only ()
→ Search returned 3793 chars
Actual route: search_only / 
Answer preview: # Fine-Tuning Explained

Based on the provided context, **fine-tuning is the process of tailoring a pre-trained model to specific needs** (page 356).

Here are the key points:

## What It Is
Fine-tuning adapts an already-trained model for your partic...

Q: What can you do?
Expected route: chat
→ Classified as: chat ()
Actual route: chat / 
Answer preview: I'm an AI assistant, so I can help with a pretty wide range of things:

**Writing

### Summary

* When did the classifier misroute? What triggered the misroute?
    tested 8 cases to stress the classifier. Some of the casual queries with technical content ("Do you know anything about RAG?") sometimes classified as chat despite being factual.
    Root cause of the compound-query misroute: my classifier prompt only allows one analysis_type at a time. If the user wants both, I lose half the intent. Fix would be to allow a list of analysis types.

* Compare Day 3's LangGraph agent (tool delegation) vs Day 5's routed agent (explicit control). Which would you deploy in production, and for what use case?
    Delegation is flexible but unpredictable.
    Explicit control is rigid but autidtable.
    Tool delegation means the model decides which tools to call and when — my code just executes its choices. Explicit control means my code decides the routing based on a classifier's output, and each node runs deterministically. Delegation is flexible; explicit control is auditable. I'd deploy delegation for research and code assistants; explicit control for anything with compliance requirements or cost routing needs.

* What's the cost tradeoff of explicit routing? (Hint: an extra classifier call per query.)
    ## Cost tradeoffs of explicit routing
- Direct: one extra classifier call, ~$0.0003 per query
- Latency: +300-500ms per request, always paid
- Failure mode: classifier misroute wastes downstream work
- Complexity: more code, second prompt-engineering surface

## When each wins
Delegation wins: open-ended queries, low cost per query, no compliance 
needs, prototype stage, adaptability > consistency.

Explicit wins: cost routing, compliance/audit, defensive routing on 
costly failures, stable flow, team growth.

## Migration path
Start with delegation. Migrate to explicit when you find yourself 
writing "always/never" rules in the system prompt to force behavior — 
that's the signal explicit routing would be cleaner.